# Планы выполнения и оптимизация Greenplum — 30 заданий

Оптимизация начинается не с переписывания SQL, а с измерения. В модуле используются `greenplum_training.opt_*` и учебные копии `m_razhin`; `EXPLAIN ANALYZE` не запускается на опасных DML.

## Результаты обучения

После **Планы и Motion** вы должны объяснять физическое выполнение на coordinator/segments, связывать logical SQL с Motion/I/O/skew, выбирать дизайн по workload и доказывать решение измерениями.

## Ментальная модель

План Greenplum — дерево локальных operators и межсегментных Motion. Redistribute меняет hash, Broadcast копирует сторону, Gather возвращает coordinator.

```text
client → coordinator (parse/optimize)
              │ dispatch slices
       ┌──────┼──────┐
       ▼      ▼      ▼
    segment segment segment
       └── Motion/interconnect ──┘
              │
              ▼
          coordinator
```
Coordinator не должен становиться местом обработки всех строк. Хороший план оставляет
scan/aggregate на сегментах и перемещает только необходимое.

## Данные и grain

Metrica и контролируемые таблицы с разным distribution. Полные схемы находятся в `data-catalog`. Общие external/raw объекты читаются, учебные результаты создаются только в `m_razhin`.

## Инженерный алгоритм

1. Назовите grain и ключ. 2. Оцените объём/cardinality. 3. Выберите distribution/storage/partition. 4. Предскажите Motion и I/O. 5. Создайте минимальный объект. 6. ANALYZE. 7. Снимите EXPLAIN и сегментные метрики. 8. Сверьте результат.

Сначала сравните estimated/actual rows, затем найдите Motion и объём строк через него; исправляйте статистику, distribution или форму JOIN по одной причине.

## Типичные ошибки

- Переносить правила PostgreSQL без учёта MPP.
- Выбирать distribution key только по высокой cardinality.
- Путать partitioning с distribution.
- Считать Broadcast всегда плохим, а Redistribute всегда допустимым.
- Сравнивать время единственного запуска без rows/Motion/I/O.
- Создавать external object с путём, доступным Windows, но не сегментам.

## Вопросы для самопроверки

1. Где физически лежит строка? 2. Какие slices выполнят сегменты? 3. Что и сколько передаёт Motion? 4. Как проявится skew? 5. Что произойдёт при повторной загрузке? 6. Как доказать результат из независимого источника?

In [ ]:
%load_ext sql
%config SqlMagic.displaylimit = 200
%sql postgresql+psycopg2://gpadmin@cbdb-coordinator:5432/moex

## 1. Что делает optimizer

Optimizer получает логическое дерево запроса и выбирает физический план: порядок JOIN, методы scan/join/aggregate, места Motion и параллельные slices. Он не знает будущее — решения основаны на статистике и cost model.

## 2. Cost не равен миллисекундам

`cost=startup..total` — условные единицы для сравнения вариантов внутри optimizer. Они учитывают оценки CPU, I/O и сети, но не являются прогнозом времени. Сравнивайте cost между планами одного окружения, а реальность — через actual time.

## 3. Estimated и actual rows

Ошибка кардинальности распространяется вверх по дереву. Если после фильтра ожидается 10 строк, а приходит миллион, optimizer может выбрать Nested Loop, неверный join order или broadcast. Удобная симметричная метрика — `max(est/actual, actual/est)`.

## 4. EXPLAIN и EXPLAIN ANALYZE

`EXPLAIN` только планирует. `EXPLAIN ANALYZE` выполняет запрос и добавляет actual rows/time/loops. Для INSERT/UPDATE/DELETE это означает реальное изменение; безопасно анализировать DML внутри `BEGIN ... ROLLBACK` либо на учебной копии.

### Как читать план

Читайте снизу вверх и для каждого узла отвечайте: сколько строк вошло, сколько вышло, где фильтр, сколько loops, есть ли Motion, совпала ли оценка, какой узел определяет critical path. В MPP дополнительно смотрите различия между сегментами.

## 5. Scan

Sequential Scan читает доступные блоки и нормален для большой доли аналитической таблицы. Index Scan полезен для селективного точечного доступа, но индекс не отменяет Motion и skew. Для AO column важна column projection.

## 6. Hash Join

Build side превращается в hash table, probe side ищет совпадения. Метод хорош для equality join. Если строки не colocated, до join появляется Redistribute/Broadcast. Недооценка build side может привести к памяти и spill.

## 7. Nested Loop

Для каждой строки внешнего набора выполняется внутренний доступ. Это отлично при нескольких внешних строках и дешёвом lookup, но катастрофично при большой ошибке cardinality. Всегда умножайте actual rows на loops.

## 8. Join order

Inner joins обычно можно переставлять, outer joins ограничивают свободу. Выгодно рано уменьшить набор селективным фильтром, но только если optimizer правильно оценил селективность. CTE и подзапрос не обязаны фиксировать порядок выполнения.

## 9. Semi и anti join

`EXISTS` спрашивает только о наличии и не размножает левую строку. `NOT EXISTS` корректно выражает отсутствие. `NOT IN` при NULL может дать UNKNOWN для всех строк — это семантика, не только производительность.

## 10. Motion как часть плана

Redistribute меняет хеш-размещение, Broadcast копирует набор, Gather собирает. Оценивайте не число надписей Motion, а объём и ширину строк. Один Motion после сильной агрегации может быть дешевле отсутствия ранней агрегации.

## 11. Двухфазная агрегация

Сегменты сначала считают partial aggregates локально, затем пересылают компактные состояния и выполняют final aggregate. Эффект максимален, если число групп намного меньше числа исходных строк.

## 12. Sort и top-N

Глобальный ORDER BY требует согласовать порядок между сегментами. Полная сортировка хранит/проливает весь набор. `ORDER BY ... LIMIT N` может использовать локальный top-N на сегментах и объединить небольшие кандидаты.

## 13. Memory и spill

Hash, aggregate и sort получают память согласно resource management и statement memory. При нехватке данные пишутся во временные файлы. Spill не всегда ошибка, но большой spill часто означает неверную оценку, слишком широкий набор или отсутствие раннего сокращения.

## 14. Статистика

`ANALYZE` собирает n_distinct, null fraction, most common values, histogram и другие данные. Statistics target управляет детализацией. После крупной загрузки статистику обновляют; чрезмерный target увеличивает время ANALYZE и каталог.

## 15. Коррелированные колонки

Обычная статистика по одной колонке не понимает зависимости вроде country→city. Комбинация предикатов может оцениваться как независимая и давать большую ошибку. Решение зависит от возможностей версии: расширенная статистика, физическая модель или переписывание.

## 16. Partition pruning

Pruning уменьшает число scan nodes/leaf. Это не замена фильтру и не distribution. Выражение над partition key, несовместимый тип или неизвестное во время планирования значение могут ограничить pruning.

## 17. Метод оптимизации

1. Зафиксировать SQL и параметры. 2. Получить plan/actual. 3. Найти крупнейшую ошибку rows. 4. Найти большой Motion/spill/skew. 5. Исправить одну причину. 6. ANALYZE при необходимости. 7. Повторить тем же способом. 8. Проверить корректность результата.

## 18. Анти-паттерны

Не отключайте planner methods как постоянное лечение. Не сравнивайте запросы с разным кэшем единственным запуском. Не увеличивайте память без границ. Не убирайте Motion ценой сильного storage skew. Не применяйте `EXPLAIN ANALYZE` к изменяющему production SQL без rollback.

### Задание 1. `m_razhin.gpo_01_explain`

**Что сделать:** Сохраните текст EXPLAIN простого фильтра и выделите scan node.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Используйте учебную функцию explain_lines.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_01_explain или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',1);

### Задание 2. `m_razhin.gpo_02_analyze`

**Что сделать:** Сравните estimated_rows и actual_rows фильтра.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

EXPLAIN ANALYZE действительно выполняет запрос.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_02_analyze или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',2);

### Задание 3. `m_razhin.gpo_03_error_ratio`

**Что сделать:** Рассчитайте cardinality_error_ratio.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

max(est/actual,actual/est).

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_03_error_ratio или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',3);

### Задание 4. `m_razhin.gpo_04_no_stats`

**Что сделать:** Создайте копию без ANALYZE и сохраните ошибку оценки.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Не запускайте ANALYZE автоматически.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_04_no_stats или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',4);

### Задание 5. `m_razhin.gpo_05_with_stats`

**Что сделать:** Выполните ANALYZE копии и повторите измерение.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Сравнивайте тот же предикат.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_05_with_stats или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',5);

### Задание 6. `m_razhin.gpo_06_column_stats`

**Что сделать:** Создайте VIEW статистики country/hot_key из pg_stats.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Покажите n_distinct, null_frac, most_common_vals/freqs.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_06_column_stats или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',6);

### Задание 7. `m_razhin.gpo_07_stats_target`

**Что сделать:** Повышайте statistics target hot_key и повторите ANALYZE.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Проверьте размер MCV списка.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_07_stats_target или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',7);

### Задание 8. `m_razhin.gpo_08_seq_scan`

**Что сделать:** Объясните и зафиксируйте scan большого диапазона.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Для аналитики seq scan часто оптимален.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_08_seq_scan или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',8);

### Задание 9. `m_razhin.gpo_09_projection`

**Что сделать:** Сравните план/байты SELECT двух колонок и SELECT *.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Особенно важно для AO column.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_09_projection или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',9);

### Задание 10. `m_razhin.gpo_10_filter_pushdown`

**Что сделать:** Покажите, где применяется фильтр относительно Motion.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Фильтр должен сокращать набор до сети.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_10_filter_pushdown или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',10);

## Уровень 2 — JOIN и Motion

### Задание 11. `m_razhin.gpo_11_hash_join`

**Что сделать:** Получите Hash Join двух наборов и сохраните показатели.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Равенство и достаточная память благоприятны hash join.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_11_hash_join или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',11);

### Задание 12. `m_razhin.gpo_12_nested_loop`

**Что сделать:** Создайте небольшой сценарий Nested Loop и объясните его уместность.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Малый внешний набор — нормальный случай.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_12_nested_loop или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',12);

### Задание 13. `m_razhin.gpo_13_bad_nested_loop`

**Что сделать:** Воспроизведите дорогой Nested Loop и перепишите запрос.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Смотрите rows loops.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_13_bad_nested_loop или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',13);

### Задание 14. `m_razhin.gpo_14_broadcast`

**Что сделать:** Измерьте Broadcast Motion маленького измерения.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Сравните размер передаваемого набора.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_14_broadcast или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',14);

### Задание 15. `m_razhin.gpo_15_redistribute`

**Что сделать:** Измерьте Redistribute Motion несовместимого JOIN.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Зафиксируйте ключ redistribution.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_15_redistribute или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',15);

### Задание 16. `m_razhin.gpo_16_colocated`

**Что сделать:** Перестройте физическую копию для colocated JOIN.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Сравните Motion и total time.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_16_colocated или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',16);

### Задание 17. `m_razhin.gpo_17_join_order`

**Что сделать:** Сравните два порядка JOIN трёх таблиц.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Optimizer может переставлять inner joins.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_17_join_order или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',17);

### Задание 18. `m_razhin.gpo_18_semi_join`

**Что сделать:** Сравните EXISTS и JOIN+DISTINCT.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Semi join не размножает строки.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_18_semi_join или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',18);

### Задание 19. `m_razhin.gpo_19_anti_join`

**Что сделать:** Реализуйте anti join через NOT EXISTS и изучите план.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Остерегайтесь NOT IN с NULL.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_19_anti_join или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',19);

### Задание 20. `m_razhin.gpo_20_skew_join`

**Что сделать:** Измерьте разброс actual rows/времени сегментов skewed join.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Один content может определять всё время.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_20_skew_join или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',20);

## Уровень 3 — aggregation, sort, spill и итоговая оптимизация

### Задание 21. `m_razhin.gpo_21_two_stage_agg`

**Что сделать:** Найдите local и final aggregation stages.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Частичная агрегация уменьшает Motion.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_21_two_stage_agg или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',21);

### Задание 22. `m_razhin.gpo_22_group_key`

**Что сделать:** Сравните GROUP BY distribution key и чужой key.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Смотрите Motion между стадиями.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_22_group_key или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',22);

### Задание 23. `m_razhin.gpo_23_distinct`

**Что сделать:** Изучите план count(distinct user_id).

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Distinct часто требует перераспределения.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_23_distinct или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',23);

### Задание 24. `m_razhin.gpo_24_sort`

**Что сделать:** Создайте сортировку большого набора и зафиксируйте memory/disk.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

ORDER BY без LIMIT требует global order.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_24_sort или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',24);

### Задание 25. `m_razhin.gpo_25_top_n`

**Что сделать:** Перепишите сортировку для top-N и сравните план.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Top-N heap может уменьшить память.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_25_top_n или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',25);

### Задание 26. `m_razhin.gpo_26_spill`

**Что сделать:** Воспроизведите spill при малой statement memory.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Меняйте параметры только локально.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_26_spill или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',26);

### Задание 27. `m_razhin.gpo_27_no_spill`

**Что сделать:** Устраните spill изменением запроса или памяти.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Большая память — не единственное решение.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_27_no_spill или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',27);

### Задание 28. `m_razhin.gpo_28_partition_plan`

**Что сделать:** Сравните план с pruning и без него.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Считайте реально просканированные leaf.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_28_partition_plan или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',28);

### Задание 29. `m_razhin.gpo_29_metrica_query`

**Что сделать:** Оптимизируйте типовой запрос Метрики по четырём предикатам.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Измерьте pruning, Motion, rows и время.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_29_metrica_query или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',29);

### Задание 30. `m_razhin.gpo_30_report`

**Что сделать:** Создайте итоговый VIEW before/after: rows error, motion, spill, time, verdict.

Сохраните измеримые признаки плана в объекте с указанным именем. До изменения сформулируйте гипотезу, после — сравните тем же запросом и подтвердите равенство результата.

<details><summary>Подсказка</summary>

Все выводы должны иметь измерение.

</details>

<!-- task-card-gp-v1 -->
#### Карточка выполнения

- **Учебная цель:** назовите распределённый механизм, который демонстрирует задание.
- **Вход/выход:** зафиксируйте grain, ключ, объём и владельца объекта.
- **Физический прогноз:** distribution, partitions, scan, Motion и ожидаемый bottleneck.
- **Порядок:** профиль → DDL/SQL → ANALYZE → EXPLAIN/метрики → reconciliation → checker.
- **Граничные случаи:** пустой вход, NULL ключа, heavy hitter, повтор запуска и недоступный segment source.
- **Checker:** прочитайте название каждой проверки и объясните, какой инвариант она доказывает.

Подсказка задаёт направление исследования, но не готовое распределение, DDL или запрос.

In [ ]:
%%sql
SET search_path TO m_razhin,public;
-- Создайте m_razhin.gpo_30_report или выполните требуемый эксперимент.

In [ ]:
%%sql
-- Ручная проверка результата и EXPLAIN/EXPLAIN ANALYZE.

In [ ]:
%%sql
SELECT * FROM greenplum_training.run_checks('query_optimization',30);

## Прогресс

In [ ]:
%%sql
SELECT * FROM greenplum_training.progress WHERE module_name='query_optimization' ORDER BY task_no;